# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.12529568 -0.68738792  0.71933295 -0.85924188  0.18396854]
 [-0.0652836  -0.13021115  0.73464695  0.51817801 -0.50900093]
 [-0.02636294  0.66906536 -0.02631529 -0.5549421   0.3334178 ]
 [ 0.67624905 -0.39219891  0.04778162 -0.71305225 -0.85392895]
 [ 0.52777692  0.28369867  0.32040398 -0.03960107  0.40000039]
 [ 0.40578955 -0.73424499 -0.98035545 -0.58051811  0.61973422]
 [-0.99110179  0.28300239  0.45452043  0.88161478  0.69143534]
 [ 0.3888662  -0.80648979 -0.53446757  0.55607887  0.97973071]
 [ 0.02107726  0.5387765  -0.86000228 -0.82684561 -0.35278368]
 [ 0.38065555 -0.72183444  0.20654791 -0.74225793 -0.53430299]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 1, 0, 0, 1, 0, 0, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:00<00:28,  1.15it/s]

SVI:   3%|▎         | 1/34 [00:00<00:28,  1.15it/s, loss=2253.0498]

SVI:   6%|▌         | 2/34 [00:00<00:27,  1.15it/s, loss=2138.2097]

SVI:   9%|▉         | 3/34 [00:00<00:26,  1.15it/s, loss=2673.3547]

...

SVI:  84%|████████▍ | 21/25 [00:01<00:03,  1.03it/s, loss=2366.5078]

SVI:  88%|████████▊ | 22/25 [00:01<00:02,  1.03it/s, loss=1986.9104]

SVI:  92%|█████████▏| 23/25 [00:01<00:01,  1.03it/s, loss=1932.6750]

SVI:  96%|█████████▌| 24/25 [00:01<00:00,  1.03it/s, loss=2040.4290]

SVI: 100%|██████████| 25/25 [00:01<00:00,  1.03it/s, loss=2587.9263]